In [ ]:
# Célula 0 — Instalar bibliotecas necessárias
!pip install scikit-learn

In [ ]:
# Celula 1 — Importar bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor #tentar aplicar!


print("Bibliotecas importadas com sucesso!")

In [ ]:
# Célula 2 — Carregamento do dataset

df_raw = pd.read_csv("br_seeg_emissoes_brasil.csv")
print(f"Shape original: {df_raw.shape}")
df_raw.head()

In [ ]:
# Célula 3 — Seleção das colunas relevantes
'''
    Tudo junto numa célula só para evitar problemas de re-execução.
    Sempre partimos do df_raw (original) para garantir consistência.
'''

# Passo 1: Selecionar colunas relevantes
df =df_raw[["ano", "nivel_1" , "gas" , "emissao"]].copy()
print(f"1- Seleção de colunas: {df.shape}")

# Passo 2: Filtrar para apenas emissão de CO2
df = df[df["gas"] == "CO2"].copy()
print(f"\n2- Filtrar para CO2: {df.shape}")

# Passo 3: verificar valores nulos
print(f"\n3- Valores nulos antes da limpeza:")
print(df.isnull().sum())

# Passo 4: Remover linhas onde "emissão" é nulo
# Modelos de ML não aceitam NaN no target (y).
df = df.dropna(subset=["emissao"])

# Passo 5: Remover coluna "gas"
#   Já filtramos só CO2 (t), então essa coluna tem valor único
#   e causaria erro "Could not convert string to float"
df = df.drop(columns=["gas"])
print(f"\n5 - Após remover coluna 'gas': {df.shape}")

# Passo 6: One-Hot Encoding na coluna 'nivel_1'
#   Converte categorias em colunas binárias (0/1)
#   drop_first=True evita multicolinearidade
df = pd.get_dummies(df, columns=["nivel_1"], drop_first=True, dtype=int)
print(f"\n6 - Após One-Hot Encoding: {df.shape}")
print(f"   Colunas {list(df.columns)}")

# verificação final
print(f"\n{'='*50}")
print(f"   Dados prontos!")
print(f"   Total de amostras: {df.shape[0]}")
print(f"   Total de colunas: {df.shape[1]}")
print(f"   'emissao' presente: {df.isnull().sum().sum()}")
print(f"\n{'='*50}")
df

In [ ]:
# Célula 4 — Separar X (features) e y (target)

# X = tudo que o modelo usa para prever
# y = o que queremos prever (emissão de CO2)

X = df.drop(columns=["emissao"])
y = df["emissao"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures em (X): {list(X.columns)}")
print(f"\ny (primeiros 5 valores): {y.head().tolist()}")
print(y.head())

# Verificações de segurança
assert X.shape[0] == y.shape[0], "X e y com tamanhos diferentes!"
assert y.shape[0] > 0, "y está vazio!"
assert X.shape[1] > 0, "X não tem features!"
print(f"\n {X.shape[0]} amostras, {X.shape[1]} features!")


In [ ]:
# Célula 5 - Normalização Min-Max

'''
    Transforma todos os valores para o intervalo [0, 1].
    Fórmula: x_norm = (x - x_min) / (x_max - x_min)
    
    Por que normalizar?
        - SVR é muito sensível à escala dos dados
        - Facilita comparação entre features
        - Melhora convergência de alguns algoritmos
        
'''
# Normalizar X (features)

scaler_X = MinMaxScaler()
X = pd.DataFrame(
    scaler_X.fit_transform(X),
    columns=X.columns,
    index=X.index
)
# Normalizar y (target) - separado para poder inverter depois
scaler_y = MinMaxScaler()
y = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index,
    name="emissao"
)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nX range: [{X.min().min():.4f}, {X.max().max():.4f}]")
print(f"y range: [{y.min()}]")